# 1. Create Human Coding Sample From Filtered Texts

This notebook creates the human-coding sample used for LLM prompt development. It starts from the already ModernBERT-filtered corpus in `filtered_texts/`, recreates or reuses the 3,000-document sample, copies those reduced texts into `training_data_database_identification/Output`, and creates a human coding workbook/codebook.

The intended downstream logic is: code the first 500 sampled texts, export the completed labels, split those 500 labels into development/test files, then continue with notebook `2` for LLM prompt development and evaluation.

In [ ]:
# Resolve the repository and external data directory without machine-specific paths.
from pathlib import Path
import sys

for _candidate in (Path.cwd(), *Path.cwd().parents):
    if (_candidate / "project_config.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError("Run this notebook from within the cloned repository.")

from project_config import REPO_ROOT, DATA_ROOT, require_env

from pathlib import Path
import csv
import html
import math
import random
import re
import shutil
from datetime import datetime

import pandas as pd

# Code is run from the OneDrive project code folder.
CODE_ROOT = REPO_ROOT
TEXT_MINING_CODE_ROOT = CODE_ROOT / "text_mining"

# Data is read from and written to the share-drive project folder.
SHARE_ROOT = DATA_ROOT
# DATA_ROOT is configured through project_config.
ROOT = DATA_ROOT

FILTERED_TEXT_ROOT = DATA_ROOT / "filtered_texts"
SAMPLE_ROOT = DATA_ROOT / "training_data_database_identification"
SAMPLE_TEXT_DIR = SAMPLE_ROOT / "Output"
HUMAN_CODING_DIR = SAMPLE_ROOT / "human_coding_package"
REVIEW_BATCHES_DIR = HUMAN_CODING_DIR / "review_batches"
LABELLED_DIR = SAMPLE_ROOT / "labelled_data"

TRAINING_SET_MANIFEST_XLSX = SAMPLE_ROOT / "training_set_dois.xlsx"
TRAINING_SET_MANIFEST_CSV = SAMPLE_ROOT / "training_set_dois.csv"
CODING_TEMPLATE_CSV = HUMAN_CODING_DIR / "database_coding_template.csv"
CODING_TEMPLATE_XLSX = HUMAN_CODING_DIR / "database_coding_template.xlsx"
CODEBOOK_FILE = HUMAN_CODING_DIR / "human_coding_codebook.md"
REVIEW_INDEX_FILE = HUMAN_CODING_DIR / "review_index.html"

COMPLETED_CODING_FILE = HUMAN_CODING_DIR / "database_coding_from_html.csv"
LABELLED_500_CSV = LABELLED_DIR / "database_coding_from_html_1-500.csv"
LABELLED_500_XLSX = LABELLED_DIR / "database_coding_from_html_1-500.xlsx"
SPLIT_MANIFEST_FILE = LABELLED_DIR / "labelled500_development_test_split.csv"
DEVELOPMENT_FILE = LABELLED_DIR / "labelled500_development400.csv"
TEST_FILE = LABELLED_DIR / "labelled500_test100.csv"

SAMPLE_SIZE = 3000
SAMPLE_SEED = 44
BATCH_SIZE = 100

# If the old 3,000-document sample manifest exists, reuse it and copy the filtered versions
# of those same documents. If it does not exist, draw a new reproducible sample from filtered_texts.
REUSE_EXISTING_SAMPLE_MANIFEST = True
OVERWRITE_SAMPLE_TEXTS = False

print(f"Filtered text root: {FILTERED_TEXT_ROOT}")
print(f"Sample text folder: {SAMPLE_TEXT_DIR}")
print(f"Human coding folder: {HUMAN_CODING_DIR}")

In [ ]:
def doi_from_filtered_filename(filename):
    safe_name = Path(filename).stem
    safe_name = re.sub(r" \(\d+\)$", "", safe_name)
    return safe_name.replace("_", "/")


def read_table(path):
    path = Path(path)
    if path.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    return pd.read_csv(path, encoding="utf-8-sig")


def write_table(df, csv_path=None, xlsx_path=None):
    if csv_path is not None:
        csv_path = Path(csv_path)
        csv_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(csv_path, index=False, encoding="utf-8-sig")
        print(f"Saved CSV: {csv_path}")

    if xlsx_path is not None:
        xlsx_path = Path(xlsx_path)
        xlsx_path.parent.mkdir(parents=True, exist_ok=True)
        try:
            df.to_excel(xlsx_path, index=False)
            print(f"Saved Excel: {xlsx_path}")
        except ImportError:
            print(f"No Excel writer is installed; skipped Excel output: {xlsx_path}")


def collect_filtered_texts():
    if not FILTERED_TEXT_ROOT.exists():
        raise FileNotFoundError(f"Filtered text root does not exist: {FILTERED_TEXT_ROOT}")

    rows = []
    for source_folder in sorted(FILTERED_TEXT_ROOT.glob("publications_*")):
        if not source_folder.is_dir():
            continue
        for txt_file in sorted(source_folder.glob("*.txt")):
            rows.append({
                "filename": txt_file.name,
                "doi": doi_from_filtered_filename(txt_file.name),
                "source_folder": source_folder.name,
                "file_path": txt_file,
            })

    if not rows:
        raise FileNotFoundError(f"No filtered .txt files found under {FILTERED_TEXT_ROOT}")

    return pd.DataFrame(rows)


def load_existing_sample_manifest(all_filtered_texts):
    if TRAINING_SET_MANIFEST_XLSX.exists():
        manifest = pd.read_excel(TRAINING_SET_MANIFEST_XLSX)
    elif TRAINING_SET_MANIFEST_CSV.exists():
        manifest = pd.read_csv(TRAINING_SET_MANIFEST_CSV, encoding="utf-8-sig")
    else:
        return None

    if "filename" not in manifest.columns or "doi" not in manifest.columns:
        raise ValueError("Existing sample manifest must contain at least filename and doi columns.")

    manifest = manifest.copy()
    if "source_folder" not in manifest.columns:
        filename_counts = all_filtered_texts["filename"].value_counts()
        duplicated_filenames = set(filename_counts[filename_counts > 1].index)
        if any(filename in duplicated_filenames for filename in manifest["filename"].astype(str)):
            raise ValueError("Cannot infer source_folder because at least one sampled filename appears in multiple filtered-text folders.")
        source_lookup = all_filtered_texts.set_index("filename")["source_folder"].to_dict()
        manifest["source_folder"] = manifest["filename"].map(source_lookup)

    manifest = manifest[["filename", "doi", "source_folder"]].copy()
    manifest["filename"] = manifest["filename"].astype(str)
    manifest["doi"] = manifest["doi"].astype(str)
    manifest["source_folder"] = manifest["source_folder"].astype(str)

    path_lookup = all_filtered_texts.set_index(["source_folder", "filename"])["file_path"].to_dict()
    manifest["file_path"] = manifest.apply(
        lambda row: path_lookup.get((row["source_folder"], row["filename"])),
        axis=1,
    )

    missing = manifest[manifest["file_path"].isna()].copy()
    if not missing.empty:
        preview = missing[["source_folder", "filename"]].head(10).to_dict(orient="records")
        raise FileNotFoundError(f"Existing manifest includes files not found in filtered_texts. First missing rows: {preview}")

    manifest["sample_source"] = "existing_manifest"
    return manifest


def build_sample_manifest():
    all_filtered_texts = collect_filtered_texts()
    print(f"Filtered text files available: {len(all_filtered_texts):,}")

    sample_df = None
    if REUSE_EXISTING_SAMPLE_MANIFEST:
        sample_df = load_existing_sample_manifest(all_filtered_texts)
        if sample_df is not None:
            print(f"Reusing existing sample manifest with {len(sample_df):,} rows.")

    if sample_df is None:
        sample_size_actual = min(SAMPLE_SIZE, len(all_filtered_texts))
        sampled_rows = random.Random(SAMPLE_SEED).sample(
            all_filtered_texts.to_dict(orient="records"),
            sample_size_actual,
        )
        sample_df = pd.DataFrame(sampled_rows)
        sample_df["sample_source"] = f"random_seed_{SAMPLE_SEED}"
        print(f"Created new sample with {len(sample_df):,} filtered texts.")

    if sample_df["filename"].duplicated().any():
        duplicated = sample_df.loc[sample_df["filename"].duplicated(), "filename"].head(10).tolist()
        raise ValueError(f"Sample contains duplicate filenames that would overwrite each other in Output/: {duplicated}")

    return sample_df.reset_index(drop=True)

In [ ]:
# =========================================================
# CREATE OR RECREATE THE 3,000-DOCUMENT FILTERED-TEXT SAMPLE
# =========================================================
sample_df = build_sample_manifest()

SAMPLE_ROOT.mkdir(parents=True, exist_ok=True)
SAMPLE_TEXT_DIR.mkdir(parents=True, exist_ok=True)

copied = 0
kept_existing = 0
for row in sample_df.itertuples(index=False):
    src = Path(row.file_path)
    dst = SAMPLE_TEXT_DIR / row.filename
    if dst.exists() and not OVERWRITE_SAMPLE_TEXTS:
        kept_existing += 1
        continue
    shutil.copy2(src, dst)
    copied += 1

manifest_df = sample_df[["filename", "doi", "source_folder"]].copy()
write_table(manifest_df, csv_path=TRAINING_SET_MANIFEST_CSV, xlsx_path=TRAINING_SET_MANIFEST_XLSX)

print(f"Sample rows: {len(manifest_df):,}")
print(f"Filtered texts copied: {copied:,}")
print(f"Existing sample texts kept: {kept_existing:,}")
print(f"Sample text folder: {SAMPLE_TEXT_DIR}")
manifest_df.head()

## Create Human Coding Workbook And Codebook

This section creates a coding workbook from the reduced sample texts. The workbook is the main hand-coding file; the HTML pages are optional reading aids that split the 3,000 texts into batches of 100.

In [ ]:
def read_sample_text(filename):
    file_path = SAMPLE_TEXT_DIR / str(filename)
    if not file_path.exists():
        raise FileNotFoundError(f"Missing sampled reduced text file: {file_path}")
    return file_path.read_text(encoding="utf-8", errors="replace").strip()


def build_coding_records():
    manifest = read_table(TRAINING_SET_MANIFEST_XLSX)
    required = {"filename", "doi", "source_folder"}
    missing = sorted(required - set(manifest.columns))
    if missing:
        raise ValueError(f"Sample manifest is missing expected columns: {missing}")

    records = manifest[["filename", "doi", "source_folder"]].copy()
    records.insert(0, "record_id", range(1, len(records) + 1))
    records["batch_id"] = ((records["record_id"] - 1) // BATCH_SIZE) + 1
    records["text"] = records["filename"].apply(read_sample_text)
    records["db_use"] = ""
    records["databases"] = ""
    records["comments"] = ""
    records["done"] = ""
    return records


def write_human_codebook():
    text = """# Human Coding Codebook: External Digital Data Resource Use

Code each sampled text for whether the study uses an external digital data resource.

## Columns

- record_id: sequential row number in the 3,000-document sample.
- filename: reduced text filename used by the LLM prompt notebook.
- doi: article DOI.
- source_folder: original filtered_texts/publications_* folder.
- text: ModernBERT-filtered text shown to the coder.
- db_use: enter 1 if the text explicitly describes use of an external digital data resource; enter 0 otherwise.
- databases: semicolon-separated names of used external data resources. Leave blank if db_use is 0 or if use is clear but no resource is named.
- comments: optional notes for ambiguity or adjudication.
- done: enter 1 when the row has been coded.

## Positive examples

Code db_use = 1 when authors retrieved, downloaded, queried, accessed, obtained, compiled from, extracted from, reused from, mapped to, interpolated from, or directly compared against the contents of an external digital data resource.

## Negative examples

Code db_use = 0 for software/tools only, instruments/sensors only, field sites only, supplementary files only, generic statements about data with no external resource use, or current-study data depositions with no external data reuse.
"""
    CODEBOOK_FILE.parent.mkdir(parents=True, exist_ok=True)
    CODEBOOK_FILE.write_text(text, encoding="utf-8")
    print(f"Saved codebook: {CODEBOOK_FILE}")


def render_batch_page(batch_df, total_batches):
    batch_id = int(batch_df["batch_id"].iloc[0])
    rows_html = []
    for row in batch_df.itertuples(index=False):
        rows_html.append(f"""
        <section class="record" id="record-{int(row.record_id):04d}">
          <h2>{int(row.record_id):04d}. {html.escape(str(row.doi))}</h2>
          <p class="meta">{html.escape(str(row.filename))} | {html.escape(str(row.source_folder))}</p>
          <pre>{html.escape(str(row.text))}</pre>
        </section>
        """)

    prev_link = f'<a href="batch_{batch_id - 1:02d}.html">Previous batch</a>' if batch_id > 1 else '<span></span>'
    next_link = f'<a href="batch_{batch_id + 1:02d}.html">Next batch</a>' if batch_id < total_batches else '<span></span>'

    return f"""<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>Human Coding Sample Batch {batch_id:02d}</title>
<style>
body {{ font-family: Arial, sans-serif; line-height: 1.45; margin: 24px; color: #1f2937; background: #f8fafc; }}
nav {{ display: flex; justify-content: space-between; margin-bottom: 18px; }}
a {{ color: #0f766e; }}
.record {{ background: white; border: 1px solid #d1d5db; border-radius: 8px; padding: 16px; margin-bottom: 16px; }}
h1 {{ margin-top: 0; }}
h2 {{ font-size: 1rem; margin: 0 0 4px; }}
.meta {{ color: #6b7280; font-size: 0.9rem; margin: 0 0 12px; }}
pre {{ white-space: pre-wrap; font-family: Georgia, 'Times New Roman', serif; background: #f9fafb; padding: 12px; border-radius: 6px; border: 1px solid #e5e7eb; }}
</style>
</head>
<body>
<nav>{prev_link}<a href="../review_index.html">Index</a>{next_link}</nav>
<h1>Batch {batch_id:02d} of {total_batches:02d}</h1>
<p>Use the Excel/CSV coding template for labels. These pages are reading aids for the reduced texts.</p>
{''.join(rows_html)}
</body>
</html>"""


def write_review_pages(records):
    REVIEW_BATCHES_DIR.mkdir(parents=True, exist_ok=True)
    total_batches = int(records["batch_id"].max())

    links = []
    for batch_id, batch_df in records.groupby("batch_id", sort=True):
        batch_file = REVIEW_BATCHES_DIR / f"batch_{int(batch_id):02d}.html"
        batch_file.write_text(render_batch_page(batch_df, total_batches), encoding="utf-8")
        start_id = int(batch_df["record_id"].min())
        end_id = int(batch_df["record_id"].max())
        links.append(f'<li><a href="review_batches/batch_{int(batch_id):02d}.html">Batch {int(batch_id):02d}</a> ({start_id}-{end_id})</li>')

    index_html = f"""<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>Human Coding Sample Index</title>
<style>
body {{ font-family: Arial, sans-serif; line-height: 1.5; max-width: 900px; margin: 32px auto; color: #1f2937; }}
a {{ color: #0f766e; }}
</style>
</head>
<body>
<h1>Human Coding Sample</h1>
<p>This package contains {len(records):,} ModernBERT-filtered texts in batches of {BATCH_SIZE}. Use <code>{CODING_TEMPLATE_XLSX.name}</code> or <code>{CODING_TEMPLATE_CSV.name}</code> to enter labels.</p>
<ul>{''.join(links)}</ul>
</body>
</html>"""
    REVIEW_INDEX_FILE.write_text(index_html, encoding="utf-8")
    print(f"Saved review index: {REVIEW_INDEX_FILE}")
    print(f"Saved batch pages: {REVIEW_BATCHES_DIR}")

In [ ]:
# =========================================================
# BUILD HUMAN CODING PACKAGE
# =========================================================
HUMAN_CODING_DIR.mkdir(parents=True, exist_ok=True)
records_df = build_coding_records()

coding_columns = [
    "record_id",
    "filename",
    "doi",
    "source_folder",
    "text",
    "db_use",
    "databases",
    "comments",
    "done",
]
write_table(records_df[coding_columns], csv_path=CODING_TEMPLATE_CSV, xlsx_path=CODING_TEMPLATE_XLSX)
write_human_codebook()
write_review_pages(records_df)

print(f"Human coding records: {len(records_df):,}")
records_df[["record_id", "filename", "doi", "source_folder"]].head()

## After Human Coding: Export 500 Labels And Create Development/Test Split

Run this section only after the first 500 records have been coded. By default the cell is disabled so that a fresh package-generation run does not fail before coding exists. Set `RUN_LABEL_EXPORT_AND_SPLIT = True` once the completed coding export is available.

In [ ]:
RUN_LABEL_EXPORT_AND_SPLIT = False

LABELLED_SAMPLE_SIZE = 500
TEST_SIZE = 100
SPLIT_SEED = 20260529


def is_done(value):
    return str(value).strip().lower() in {"1", "1.0", "true", "yes", "done"}


def normalize_binary(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip()
    if text in {"0", "0.0"}:
        return 0
    if text in {"1", "1.0"}:
        return 1
    return pd.NA


def load_completed_coding(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Completed coding file does not exist yet: {path}")
    coded = read_table(path)
    required = {"doi", "db_use", "databases", "comments", "done"}
    missing = sorted(required - set(coded.columns))
    if missing:
        raise ValueError(f"Completed coding file is missing expected columns: {missing}")
    return coded


def export_labelled_500(coded):
    done_df = coded[coded["done"].map(is_done)].copy().reset_index(drop=True)
    if len(done_df) < LABELLED_SAMPLE_SIZE:
        raise ValueError(f"Only {len(done_df)} done rows found; expected at least {LABELLED_SAMPLE_SIZE}.")

    labelled_df = done_df.head(LABELLED_SAMPLE_SIZE).copy()
    labelled_df["db_use"] = labelled_df["db_use"].apply(normalize_binary)
    labelled_df = labelled_df[["doi", "db_use", "databases", "comments", "done"]].copy()

    write_table(labelled_df, csv_path=LABELLED_500_CSV, xlsx_path=LABELLED_500_XLSX)
    return labelled_df


def create_development_test_split(labelled_df):
    manifest = read_table(TRAINING_SET_MANIFEST_XLSX)
    manifest = manifest[["doi", "filename", "source_folder"]].copy()

    merged = labelled_df.merge(manifest, on="doi", how="left", validate="one_to_one")
    if merged["filename"].isna().any():
        missing = merged.loc[merged["filename"].isna(), "doi"].head(10).tolist()
        raise ValueError(f"Some labelled DOIs are missing from the sample manifest. First missing DOIs: {missing}")

    merged["db_use_int"] = merged["db_use"].apply(normalize_binary)
    known_df = merged[merged["db_use_int"].isin([0, 1])].copy()
    unresolved_df = merged[~merged["db_use_int"].isin([0, 1])].copy()

    if len(known_df) < TEST_SIZE:
        raise ValueError(f"Only {len(known_df)} rows have usable 0/1 labels; cannot draw a {TEST_SIZE}-row test set.")

    counts = known_df["db_use_int"].value_counts().sort_index()
    raw_allocations = counts / counts.sum() * TEST_SIZE
    allocations = raw_allocations.apply(math.floor).astype(int)
    remainder = TEST_SIZE - int(allocations.sum())
    if remainder:
        fractional_order = (raw_allocations - allocations).sort_values(ascending=False)
        for label_value in fractional_order.index[:remainder]:
            allocations.loc[label_value] += 1

    test_parts = []
    for label_value, n_rows in allocations.items():
        group = known_df[known_df["db_use_int"] == label_value]
        test_parts.append(group.sample(n=int(n_rows), random_state=SPLIT_SEED + int(label_value)))

    test_df = pd.concat(test_parts, ignore_index=True)
    test_dois = set(test_df["doi"])
    development_df = pd.concat(
        [known_df[~known_df["doi"].isin(test_dois)], unresolved_df],
        ignore_index=True,
    )

    test_df["split"] = "test"
    development_df["split"] = "development"

    split_df = pd.concat([development_df, test_df], ignore_index=True)
    split_df["evaluation_eligible"] = split_df["db_use_int"].isin([0, 1])
    split_df["split_seed"] = SPLIT_SEED
    split_df["split_notes"] = ""
    split_df.loc[~split_df["evaluation_eligible"], "split_notes"] = (
        "Placed in development because db_use is missing or not a usable 0/1 value."
    )

    output_cols = [
        "doi",
        "filename",
        "source_folder",
        "split",
        "evaluation_eligible",
        "db_use",
        "databases",
        "comments",
        "done",
        "split_seed",
        "split_notes",
    ]
    split_df = split_df[output_cols].sort_values(["split", "filename", "doi"]).reset_index(drop=True)
    development_out = split_df[split_df["split"] == "development"].reset_index(drop=True)
    test_out = split_df[split_df["split"] == "test"].reset_index(drop=True)

    write_table(split_df, csv_path=SPLIT_MANIFEST_FILE)
    write_table(development_out, csv_path=DEVELOPMENT_FILE)
    write_table(test_out, csv_path=TEST_FILE)

    print(f"Development rows: {len(development_out):,}")
    print(f"Test rows: {len(test_out):,}")
    print("Test split label counts:")
    print(test_out["db_use"].value_counts(dropna=False).sort_index())

    return split_df, development_out, test_out


if RUN_LABEL_EXPORT_AND_SPLIT:
    coded_df = load_completed_coding(COMPLETED_CODING_FILE)
    labelled_500_df = export_labelled_500(coded_df)
    split_df, development_df, test_df = create_development_test_split(labelled_500_df)
else:
    print("Label export/split is disabled. Set RUN_LABEL_EXPORT_AND_SPLIT = True after the first 500 records have been coded.")
    print(f"Expected completed coding file: {COMPLETED_CODING_FILE}")